All raw data used for the project are available for viewing at https://drive.google.com/drive/folders/1zYKFfSXptnHTim8ERtAOu1GdgunQIPzA?usp=sharing

Installations

In [6]:
pip install -U pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 55.4 MB/s eta 0:00:00


Load datasets into the notebook

In [26]:
# Mounting procedure adapted from https://colab.research.google.com/notebooks/io.ipynb#scrollTo=RWSJpsyKqHjH
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [30]:
import dask.bag as db

In [35]:
import pypdfium2 as pdfium

In [75]:
def book_loader(filename):
  document = pdfium.PdfDocument('/content/drive/My Drive/Colab Notebooks/BigDataProject/books/' + filename + '.pdf')

  version_marker_found = 0
  book_started = 0
  text = ''
  for page in document:
    # extract text from the page
    textpage = page.get_textpage()
    extractedtext = textpage.get_text_bounded()

    # the following two conditions ensure that the material attached to the book that is not a part of the original text is skipped (for example the cover page, info about publication etc.)
    if version_marker_found == 0 and 'Verze' in extractedtext:
      version_marker_found = 1
      print('version marker found')

    # after the version marker is found (indicating the last page of added material), the next page is checked for containing the contents (obsah) of the book which can also be skipped.
    elif version_marker_found and book_started == 0:
      if 'obsah' not in extractedtext.lower():
        book_started = 1

    if book_started:
      text += extractedtext

In [76]:
book_loader('babicka_bozena_nemcova')

version marker found


In [ ]:
b = db.from_sequence(['babicka_bozena_nemcova']).map(load_book)